# OCR Gate A — CRAFT emergency threshold review, 100 frames

Attach the private mini Dataset containing `ocr-gate-a-mini.zip` (SHA-256 `bfaa3fae312e17b8ba5a4698406c5b1afb4a8bdf0d1117659f242839b6d3170f`), select one **T4 GPU**, enable Internet, then Run All. Do not attach the full `thvu165/aic-2026-keyframes` Dataset. Under the explicit deadline override, the notebook verifies the original immutable 300-frame mini manifest but evaluates its pinned first 100 rows (60 V001 + 40 V002). This is not a balanced five-video PASS.

It does **not** run EasyOCR recognition, Vintern, Gemini, production batches, or select a threshold without completed human labels. Download `/kaggle/working/ocr_gate_a_review_bundle.zip`; fill its CSV while viewing the matching 2×2 review images, then evaluate locally with `scripts.evaluate_ocr_craft_gate_a`.


In [ ]:
# Preserve Kaggle's binary ABI and use GPU 0 only.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import importlib.metadata as md
import importlib.util
import subprocess
import sys

def version_or_missing(name):
    try: return md.version(name)
    except md.PackageNotFoundError: return None

protected_names = ['torch','torchvision','numpy','opencv-python','opencv-python-headless','scikit-image','pandas']
protected_before = {name: version_or_missing(name) for name in protected_names}
packages = ['easyocr==1.7.2','python-bidi==0.6.6','pyclipper==1.3.0.post6','ninja==1.11.1.4']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir','--no-deps',*packages])
missing = [name for name in ['yaml','scipy','skimage','shapely'] if importlib.util.find_spec(name) is None]
assert not missing, ('Kaggle image missing expected modules', missing)
protected_after = {name: version_or_missing(name) for name in protected_names}
assert protected_before == protected_after, ('pip changed Kaggle binary packages',protected_before,protected_after)
abi = subprocess.run([sys.executable,'-c','import numpy,cv2,pandas,skimage; print(numpy.__version__,cv2.__version__,pandas.__version__,skimage.__version__)'],text=True,capture_output=True)
assert abi.returncode == 0, 'Dirty binary ABI; start a fresh Kaggle session.\n' + abi.stderr
print('ABI_PROBE',abi.stdout.strip(),{'protected':protected_after,'easyocr':version_or_missing('easyocr')})


In [ ]:
from __future__ import annotations

import csv, hashlib, json, math, re, shutil, time, urllib.request, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

DEV_EXPECTED = {
 'L21_V001': {'count':1008,'uid_set_sha256':'d97f6d1cb014354b11942ea908b2f75fb7ec423dc44b87451b8fe157fc16eee2'},
 'L21_V002': {'count':843,'uid_set_sha256':'62773f4ecc75cac52d6df5f598734fd9f608afa3ea680a74da91b983031596ca'},
 'L21_V003': {'count':765,'uid_set_sha256':'b7d1fe33ecfeb644506b6015ee98a6e03b9b8a9baade275fab303cbe7bd4d03b'},
 'L21_V005': {'count':744,'uid_set_sha256':'2f16c1ddc51b87bfe3299f99910f01bf7e2b8ec718b6f99015cdda21982199b0'},
 'L21_V006': {'count':804,'uid_set_sha256':'b3b6e61ff01dbdef3afc2de96f8a077c992f43e32640264743b707bfe84942cc'},
}
POLICY = {
 'schema_version':2,'sample_frames':100,
 'video_ids':['L21_V001','L21_V002','L21_V003','L21_V005','L21_V006'],
 'frames_per_video':None,'sample_video_counts':{'L21_V001':60,'L21_V002':40,'L21_V003':0,'L21_V005':0,'L21_V006':0},
 'sample_uid_set_sha256':'4544608e596f9bb80d8356017ba2f5e5242717695a378566401b57d4dbb9a778','allow_current_fallback_for_gate_b':True,
 'evidence_limitations':['Emergency single-annotator review uses the already labeled first 100 deterministic Gate A rows: 60 L21_V001 and 40 L21_V002.','All 100 labeled frames contain text, so no-text false-positive rate is unavailable.','This deadline override may retain recall_current for full-dev Gate B but is not a balanced five-video threshold PASS.'],
 'min_region_recall':0.98,'min_text_frame_recall':0.99,
 'configs':[
  {'config_id':'recall_current','text_threshold':0.6,'low_text':0.3,'link_threshold':0.3},
  {'config_id':'balanced','text_threshold':0.7,'low_text':0.4,'link_threshold':0.4},
  {'config_id':'strict','text_threshold':0.8,'low_text':0.5,'link_threshold':0.5},
 ],
}
EXPECTED_POLICY_SHA256 = 'cb32f59bfacc22f8babbba65202d3dd40ce3d607c22817264e35b2a03bd9c561'
CRAFT_WEIGHT = {'url':'https://github.com/JaidedAI/EasyOCR/releases/download/pre-v1.1.6/craft_mlt_25k.zip','zip_sha256':'8dc6a1c703a89ed56308ef742d26ebd45c656248cbbbda6e7fe60e569f873e65','weight_name':'craft_mlt_25k.pth','weight_sha256':'4a5efbfb48b4081100544e75e1e2b57f8de3d84f213004b14b85fd4b3748db17'}
INPUT_ROOT=Path('/kaggle/input'); WORK=Path('/kaggle/working/ocr-gate-a-craft-v1'); MODEL_DIR=WORK/'easyocr-models'; REVIEW_DIR=WORK/'review-images'
RESULTS=WORK/'craft-threshold-results.jsonl'; SAMPLE=WORK/'sample-manifest.jsonl'; REVIEW_CSV=WORK/'craft-gate-a-human-review.csv'; REPORT=WORK/'craft-gate-a-pending-report.json'; BUNDLE=Path('/kaggle/working/ocr_gate_a_review_bundle.zip')
for path in (WORK,MODEL_DIR,REVIEW_DIR): path.mkdir(parents=True,exist_ok=True)

def canonical_json(value): return json.dumps(value,sort_keys=True,separators=(',',':')).encode('utf-8')
assert hashlib.sha256(canonical_json(POLICY)).hexdigest()==EXPECTED_POLICY_SHA256
def sha256_file(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as handle:
  for block in iter(lambda:handle.read(8*1024*1024),b''): d.update(block)
 return d.hexdigest()
def make_uid(video_id,shot_id,local_idx): return int.from_bytes(hashlib.blake2b(f'{video_id}:{shot_id}:{local_idx}'.encode(),digest_size=8).digest(),'big')>>1
def uid_set_sha256(values): return hashlib.sha256(''.join(f'{v}\n' for v in sorted(values)).encode()).hexdigest()
def append_jsonl(path,row):
 with Path(path).open('a',encoding='utf-8') as handle: handle.write(json.dumps(row,ensure_ascii=False,sort_keys=True)+'\n'); handle.flush(); os.fsync(handle.fileno())
def load_jsonl(path,key_fields):
 rows=[]; seen=set(); path=Path(path)
 if not path.exists(): return rows
 for number,line in enumerate(path.read_text(encoding='utf-8').splitlines(),1):
  if not line.strip(): continue
  row=json.loads(line); key=tuple(row[field] for field in key_fields)
  if key in seen: raise ValueError(f'duplicate {key} at line {number}')
  seen.add(key); rows.append(row)
 return rows
assert torch.cuda.is_available() and torch.cuda.device_count()==1
assert 'T4' in torch.cuda.get_device_name(0).upper(), torch.cuda.get_device_name(0)
print('GPU',torch.cuda.get_device_name(0),'POLICY_SHA256',EXPECTED_POLICY_SHA256)


In [ ]:
# Load the immutable mini Dataset without recursively scanning the full AIC Dataset.
EXPECTED_MINI_ZIP_SHA256='bfaa3fae312e17b8ba5a4698406c5b1afb4a8bdf0d1117659f242839b6d3170f'
EXPECTED_MINI_MANIFEST_SHA256='499ecd6008d38fe29c823e8c0cbb5d63530c29cfe0d9b57953113f55580e72d2'
mini_root=None
manifest_candidates=list(INPUT_ROOT.rglob('gate-a-manifest.jsonl'))
zip_candidates=list(INPUT_ROOT.rglob('ocr-gate-a-mini.zip'))
if not manifest_candidates and not zip_candidates:
 input_files=[str(path) for path in INPUT_ROOT.rglob('*') if path.is_file()]
 raise FileNotFoundError(f'Gate A mini files not found. Attached input files (first 30): {input_files[:30]}')
assert len(manifest_candidates)+len(zip_candidates)==1,('Attach exactly one Gate A mini Dataset',manifest_candidates,zip_candidates)
if zip_candidates:
 mini_zip=zip_candidates[0]
 assert sha256_file(mini_zip)==EXPECTED_MINI_ZIP_SHA256,('mini ZIP checksum mismatch',sha256_file(mini_zip))
 mini_root=WORK/'mini-input'
 if mini_root.exists(): shutil.rmtree(mini_root)
 mini_root.mkdir(parents=True)
 with zipfile.ZipFile(mini_zip) as archive:
  assert archive.testzip() is None,'corrupt mini ZIP'
  archive.extractall(mini_root)
 manifest=mini_root/'gate-a-manifest.jsonl'
else:
 manifest=manifest_candidates[0]; mini_root=manifest.parent
assert sha256_file(manifest)==EXPECTED_MINI_MANIFEST_SHA256,('mini manifest checksum mismatch',sha256_file(manifest))
mini_rows=[]
for line_number,line in enumerate(manifest.read_text(encoding='utf-8').splitlines(),1):
 if not line.strip(): continue
 row=json.loads(line); image_path=mini_root/row['image_file']
 assert image_path.is_file(),(line_number,image_path)
 assert sha256_file(image_path)==row['image_sha256'],('image checksum mismatch',line_number,image_path)
 assert row['keyframe_uid']==make_uid(row['video_id'],row['shot_id'],row['local_idx']),('UID mismatch',line_number)
 row['source_path']=str(image_path); mini_rows.append(row)
mini_rows.sort(key=lambda row:(row['video_id'],row['shot_id'],row['local_idx']))
sample=mini_rows[:POLICY['sample_frames']]
assert len(mini_rows)==300 and len(sample)==POLICY['sample_frames']==100
assert len({row['keyframe_uid'] for row in sample})==100 and len({(row['video_id'],row['shot_id']) for row in sample})==100
assert {video_id:sum(row['video_id']==video_id for row in sample) for video_id in POLICY['video_ids']}==POLICY['sample_video_counts']
assert uid_set_sha256([row['keyframe_uid'] for row in sample])==POLICY['sample_uid_set_sha256']
with SAMPLE.open('w',encoding='utf-8',newline='\n') as handle:
 for row in sample: handle.write(json.dumps(row,ensure_ascii=False,sort_keys=True)+'\n')
print('MINI_DATASET_VERIFIED',mini_root,'SAMPLE',len(sample),Counter(r['video_id'] for r in sample),'SHA256',sha256_file(SAMPLE),flush=True)


In [ ]:
# Verify the official CRAFT bytes and run each pre-registered threshold with append-safe resume.
archive=WORK/'craft_mlt_25k.zip'; weight=MODEL_DIR/CRAFT_WEIGHT['weight_name']
if not archive.exists() or sha256_file(archive)!=CRAFT_WEIGHT['zip_sha256']: urllib.request.urlretrieve(CRAFT_WEIGHT['url'],archive)
assert sha256_file(archive)==CRAFT_WEIGHT['zip_sha256']
if not weight.exists() or sha256_file(weight)!=CRAFT_WEIGHT['weight_sha256']:
 with zipfile.ZipFile(archive) as z:
  member=next(name for name in z.namelist() if name.endswith(CRAFT_WEIGHT['weight_name']))
  with z.open(member) as source,weight.open('wb') as target: shutil.copyfileobj(source,target)
assert sha256_file(weight)==CRAFT_WEIGHT['weight_sha256']
import easyocr
reader=easyocr.Reader(['en'],gpu='cuda:0',model_storage_directory=str(MODEL_DIR),download_enabled=False,detector=True,recognizer=False,verbose=True)

def quad_from_horizontal(box):
 x1,x2,y1,y2=map(float,box); return [x1,y1,x2,y1,x2,y2,x1,y2]
def detect_one(frame,config):
 image=cv2.imread(frame['source_path'],cv2.IMREAD_COLOR); assert image is not None,frame['source_path']; started=time.perf_counter()
 horizontal,free=reader.detect(image,min_size=10,text_threshold=config['text_threshold'],low_text=config['low_text'],link_threshold=config['link_threshold'],canvas_size=2560,mag_ratio=1.0,slope_ths=0.1,ycenter_ths=0.5,height_ths=0.5,width_ths=0.5,add_margin=0.1,reformat=True)
 h0=horizontal[0] if horizontal else []; f0=free[0] if free else []; boxes=[quad_from_horizontal(box) for box in h0]+[[float(v) for point in box for v in point] for box in f0]
 return {'schema_version':1,'video_id':frame['video_id'],'shot_id':frame['shot_id'],'keyframe_uid':frame['keyframe_uid'],'source_image':frame['source_image'],'config_id':config['config_id'],'thresholds':{k:config[k] for k in ('text_threshold','low_text','link_threshold')},'status':'success','error':None,'region_count':len(boxes),'boxes_px':boxes,'latency_seconds':time.perf_counter()-started}

existing=load_jsonl(RESULTS,('keyframe_uid','config_id')); done={(r['keyframe_uid'],r['config_id']) for r in existing}
for frame_index,frame in enumerate(sample,1):
 for config in POLICY['configs']:
  key=(frame['keyframe_uid'],config['config_id'])
  if key not in done: append_jsonl(RESULTS,detect_one(frame,config)); done.add(key)
 if frame_index%25==0: print('CRAFT_GATE_A_PROGRESS',frame_index,'/',len(sample))
results=load_jsonl(RESULTS,('keyframe_uid','config_id')); expected={(r['keyframe_uid'],c['config_id']) for r in sample for c in POLICY['configs']}
assert {(r['keyframe_uid'],r['config_id']) for r in results}==expected and len(results)==900
del reader; torch.cuda.empty_cache()
print('RESULTS',len(results),'SHA256',sha256_file(RESULTS))


In [ ]:
# Build one 2x2 panel/frame and an intentionally blank human-label CSV. No automatic PASS is possible here.
by_key={(r['keyframe_uid'],r['config_id']):r for r in results}; colors={'recall_current':(255,70,70),'balanced':(255,190,30),'strict':(40,220,120)}
font=ImageFont.load_default(); review_rows=[]
for index,frame in enumerate(sample,1):
 source=Image.open(frame['source_path']).convert('RGB'); panels=[]
 variants=[('ORIGINAL',None)]+[(c['config_id'],c['config_id']) for c in POLICY['configs']]
 for title,config_id in variants:
  panel=source.copy(); draw=ImageDraw.Draw(panel)
  if config_id is not None:
   result=by_key[(frame['keyframe_uid'],config_id)]; color=colors[config_id]
   for flat in result['boxes_px']:
    points=[(flat[i],flat[i+1]) for i in range(0,8,2)]; draw.line(points+[points[0]],fill=color,width=max(2,source.width//500))
   title=f'{config_id}: {result["region_count"]} regions'
  panel.thumbnail((640,360),Image.Resampling.LANCZOS); tile=Image.new('RGB',(640,390),'white'); tile.paste(panel,((640-panel.width)//2,30+(360-panel.height)//2)); ImageDraw.Draw(tile).text((8,8),title,fill='black',font=font); panels.append(tile)
 sheet=Image.new('RGB',(1280,780),'white')
 for offset,panel in enumerate(panels): sheet.paste(panel,((offset%2)*640,(offset//2)*390))
 image_name=f'{index:03d}_{frame["video_id"]}_{frame["keyframe_uid"]}.jpg'; sheet.save(REVIEW_DIR/image_name,quality=88,optimize=True)
 row={'sample_index':index,'image_file':f'review-images/{image_name}','video_id':frame['video_id'],'keyframe_uid':frame['keyframe_uid'],'shot_id':frame['shot_id'],'source_image':frame['source_image']}
 for config in POLICY['configs']: row[f'detected_regions__{config["config_id"]}']=by_key[(frame['keyframe_uid'],config['config_id'])]['region_count']
 row.update({'gt_has_text':'','gt_region_count':'',**{f'missed_gt_regions__{c["config_id"]}':'' for c in POLICY['configs']},'annotator':'','notes':''}); review_rows.append(row)

fieldnames=list(review_rows[0]);
with REVIEW_CSV.open('w',encoding='utf-8-sig',newline='') as handle:
 writer=csv.DictWriter(handle,fieldnames=fieldnames); writer.writeheader(); writer.writerows(review_rows)
instructions=WORK/'LABELING_INSTRUCTIONS.txt'
instructions.write_text('For each 2x2 image, inspect ORIGINAL and the three overlays. Fill gt_has_text=yes/no and gt_region_count as the number of real readable text regions in ORIGINAL. For each threshold, fill missed_gt_regions__<id> with how many real GT regions its overlay missed (0..gt_region_count). A box on texture/logo noise is not a hit. Fill annotator on every row. Do not change IDs or detected-region columns.\n',encoding='utf-8')
report={'schema_version':1,'created_utc':datetime.now(timezone.utc).isoformat(),'decision':'PENDING_HUMAN_GROUND_TRUTH','gate_b_allowed':False,'policy':POLICY,'policy_sha256':EXPECTED_POLICY_SHA256,'sample_manifest_sha256':sha256_file(SAMPLE),'results_sha256':sha256_file(RESULTS),'review_csv_sha256':sha256_file(REVIEW_CSV),'frames':len(sample),'results':len(results),'videos':dict(Counter(r['video_id'] for r in sample)),'model_calls':{'craft_detection':len(results),'easyocr_recognition':0,'vintern':0,'gemini':0}}
REPORT.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
with zipfile.ZipFile(BUNDLE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
 for path in (SAMPLE,RESULTS,REVIEW_CSV,REPORT,instructions): archive.write(path,path.relative_to(WORK).as_posix())
 for path in sorted(REVIEW_DIR.glob('*.jpg')): archive.write(path,path.relative_to(WORK).as_posix())
print(json.dumps(report,indent=2)); print('DOWNLOAD',BUNDLE,'BYTES',BUNDLE.stat().st_size,'SHA256',sha256_file(BUNDLE))
